<a href="https://colab.research.google.com/github/darinddv/chromatin_potential/blob/main/notebooks/simulator_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Generalized simulator — demonstration & validation

Demonstrates that ONE `Simulator` class covers every experiment in the project:

| experiment | what changes |
|---|---|
| statics (k2 reproduction) | `SAVE_MAPS_ONLY`, all three phases |
| dynamics (Koopman/DMD) | `SAVE_DYNAMICS` (positions + velocities) |
| quench (cell A -> cell B) | `initial_coords` + `phases=('production',)` + new `Model` |
| any kernel / potential | swap the `Model` — simulator untouched |

**Validation targets**
1. Regression: statics path still reproduces k2 (saddle ~2.073–2.087).
2. Pilot: is the system overdamped? (decides whether gEDMD is viable)
3. Quench: does the in-place velocity-continuous swap work, or fall back to restart?

## 0. Setup

In [ ]:
from google.colab import drive
!git clone -q https://github.com/darinddv/chromatin_potential.git
!pip install -q "openmm[cuda12]" OpenMiChroM cooler cooltools
import sys; sys.path.insert(0, '/content/chromatin_potential/src')
drive.mount('/content/drive')

import os, json, numpy as np
from collections import Counter
import matplotlib.pyplot as plt

from chromatin_potential.model import Model, _A_FULL, _L_FULL
from chromatin_potential.simulator import (
    Simulator, BackgroundStack, SaveSpec,
    SAVE_MAPS_ONLY, SAVE_DYNAMICS, SAVE_EVERYTHING,
    contact_probability, observed_over_expected, saddle_strength,
    verify_inplace_quench, swap_coupling_in_context)
print('setup OK')

In [ ]:
DATA = '/content/drive/MyDrive/uky_cheng/tecsas_ablation'
OUT  = f'{DATA}/simulator_demo'
os.makedirs(OUT, exist_ok=True)
SEQ_HG38 = f'{DATA}/chr10_beads_hg38.txt'
assert os.path.exists(SEQ_HG38), SEQ_HG38

labels = np.array([l.split()[1] for l in open(SEQ_HG38) if l.strip()])
isA = np.isin(labels, ['A1','A2']); isB = np.isin(labels, ['B1','B2','B3','B4'])
print(f'{len(labels)} beads  A={isA.sum()} B={isB.sum()}')

## 1. Build the k2 model (Part A) — unchanged from the validated run

The `Model` is the ONLY thing that determines the potential. Everything below
reuses this same object; the simulator never knows or cares which kernel made it.

In [ ]:
m5 = Model.from_type_matrix(_A_FULL, _L_FULL, real_types=[0,1,2,3,4], k=2)
A_k2 = m5.coupling_matrix()

# expand 5x5 -> 7x7 native (B4 <- B3, NA keeps original couplings)
full = _A_FULL.copy()
full[np.ix_([0,1,2,3,4],[0,1,2,3,4])] = A_k2
full[5,:5] = A_k2[4,:]; full[:5,5] = A_k2[:,4]; full[5,5] = A_k2[4,4]

ff_k2 = f'{OUT}/k2.ff'
with open(ff_k2,'w') as f:
    f.write(','.join(_L_FULL)+'\n')
    for a in range(7):
        f.write(','.join(f'{full[a,b]:.6E}' for b in range(7))+'\n')
print('k2 potential written'); print(open(ff_k2).read())

## 2. Regression: the statics path is unchanged

`SAVE_MAPS_ONLY` + all three phases = exactly the protocol that gave 2.073.
Run a couple of replicas to confirm nothing broke in the refactor.
(Full validation is 20 replicas; 2–3 is enough to catch a regression.)

In [ ]:
sim = Simulator(seq_file=SEQ_HG38, out_dir=OUT, platform='cuda',
                background=BackgroundStack())

for s in range(2):
    sim.run_replica('k2stat', s, types_table=ff_k2, save_spec=SAVE_MAPS_ONLY)

P = sim.pool('k2stat')
s_stat = saddle_strength(observed_over_expected(P), isA, isB)
print(f'\nstatics saddle (2 replicas) = {s_stat:.3f}   target ~2.07-2.09')
print('NOTE: 2 replicas is noisier than 20; a value in ~1.9-2.3 is consistent.')

### 2a. Run metadata is now recorded on every run

This is the block that makes runs reproducible and lets forces be recomputed
post hoc if they were not saved.

In [ ]:
meta = sim.load_metadata('k2stat', 0)
for k in ['compute_platform','integrator','timestep','friction_gamma','temperature',
          'particle_mass_0','n_particles','cm_motion_remover','openmm_version',
          'openmichrom_version','convergence','wall_minutes']:
    print(f'{k:22s}: {meta.get(k)}')
print('\nforces present in System:'); print(meta.get('forces'))

## 3. PILOT — the overdamped check

**This is the question that could invalidate gEDMD**, and it costs one short run.

Saves positions + velocities + forces every step, then tests whether
v ≈ F/(γm). High correlation and ratio ≈ 1 ⇒ strongly overdamped ⇒ generator-based
methods on positions alone are well founded. Low ⇒ inertia matters ⇒ need the
underdamped formulation or velocities in the state.

Short production (2000 steps) — this is a diagnostic, not an ensemble.

In [ ]:
pilot_spec = SaveSpec(contact_map=False, trajectory=True, velocities=True,
                      forces=True, interval=1, monitor=True)

sim.run_replica('pilot', 0, types_table=ff_k2,
                phases=('collapse','equil','production'),
                n_production=2000, save_spec=pilot_spec, overwrite=True)

d  = sim.load_trajectory('pilot', 0)
md_ = sim.load_metadata('pilot', 0)
x, v, f = d['xyz'], d['vel'], d['frc']
print('shapes  xyz', x.shape, ' vel', v.shape, ' frc', f.shape)

In [ ]:
# parse gamma and mass out of the recorded metadata
def _num(s):
    return float(str(s).split()[0])

gamma = _num(md_['friction_gamma'])
mass  = _num(md_['particle_mass_0'])
print(f'gamma = {gamma}, mass = {mass}')

pred = f / (gamma * mass)
r = np.corrcoef(pred.ravel(), v.ravel())[0,1]
ratio = np.linalg.norm(pred)/np.linalg.norm(v)
print(f'\ncorr(v, F/(gamma*m)) = {r:.4f}')
print(f'magnitude ratio      = {ratio:.4f}')
print(f'|F| mean/max         = {np.abs(f).mean():.4g} / {np.abs(f).max():.4g}'
      f'   finite={np.isfinite(f).all()}')
print(f'positions dtype      = {x.dtype}  finite={np.isfinite(x).all()}')

print('\n--- interpretation ---')
if r > 0.9 and 0.5 < ratio < 2.0:
    print('STRONGLY OVERDAMPED: gEDMD on positions alone is well founded.')
elif r > 0.6:
    print('PARTIALLY OVERDAMPED: usable, but check the underdamped formulation.')
else:
    print('NOT OVERDAMPED: inertia matters. Use the underdamped generator, or')
    print('include velocities in the state vector. Do NOT assume gEDMD-on-x.')

### 3a. Sanity plots for the pilot

In [ ]:
fig, ax = plt.subplots(1,3, figsize=(13,3.4))
ax[0].plot(md_['rg_trace']); ax[0].set_title('Rg trace (monitor)'); ax[0].set_xlabel('frame')
ax[1].hist(np.abs(f).ravel(), bins=60); ax[1].set_yscale('log'); ax[1].set_title('|force| distribution')
idx = np.random.default_rng(0).choice(v.size, 4000, replace=False)
ax[2].scatter(pred.ravel()[idx], v.ravel()[idx], s=2, alpha=.3)
lim = np.percentile(np.abs(v), 99.5)
ax[2].plot([-lim,lim],[-lim,lim],'r--',lw=1)
ax[2].set_xlabel('F/(gamma m)'); ax[2].set_ylabel('v'); ax[2].set_title(f'overdamped check r={r:.3f}')
plt.tight_layout(); plt.show()

## 4. Dynamics-mode run

Same simulator, different `SaveSpec`. Positions **float32** (never float16 —
you finite-difference these) plus velocities (stochastic, unrecoverable later).
Uniform interval so you can subsample any lag afterwards.

Short demo here; a real dynamics ensemble would use full production length.

In [ ]:
dyn_spec = SaveSpec(contact_map=True, trajectory=True, velocities=True,
                    forces=False, interval=100, monitor=True)

sim.run_replica('dyn', 0, types_table=ff_k2, n_production=100_000,
                save_spec=dyn_spec, overwrite=True)

d = sim.load_trajectory('dyn', 0)
print('saved arrays:', list(d.keys()))
print('xyz', d['xyz'].shape, d['xyz'].dtype, '| vel', d['vel'].shape)
print('frame_steps[:5]', d['frame_steps'][:5])

### 4a. Quick dynamics observables from the saved trajectory

These are the Di Pierro 2018 baselines the Koopman aim must reproduce first.
Computed POST HOC from saved state — nothing was accumulated during the run.

In [ ]:
xyz = d['xyz']
# MSD vs lag (single replica, demo only)
lags = np.unique(np.geomspace(1, len(xyz)//3, 12).astype(int))
msd = [np.mean(np.sum((xyz[l:]-xyz[:-l])**2, axis=-1)) for l in lags]

# velocity autocorrelation
vv = d['vel']
vac = [np.mean(np.sum(vv[t:]*vv[:len(vv)-t], axis=-1)) for t in range(0,30)]
vac = np.array(vac)/vac[0]

fig, ax = plt.subplots(1,2, figsize=(9,3.4))
ax[0].loglog(lags, msd, 'o-'); ax[0].set_xlabel('lag (frames)'); ax[0].set_ylabel('MSD')
sl = np.polyfit(np.log(lags), np.log(msd), 1)[0]
ax[0].set_title(f'MSD, slope={sl:.2f}  (2018 paper: 0.29)')
ax[1].plot(vac, 'o-'); ax[1].axhline(0, c='k', lw=.5)
ax[1].set_xlabel('lag (frames)'); ax[1].set_title('velocity autocorrelation')
plt.tight_layout(); plt.show()
print('NOTE: one short replica — indicative only, not a measurement.')

## 5. Quench: does the in-place (velocity-continuous) swap work?

Two paths:
- **restart-based** (`quench_replica`) — always works, but velocities are
  re-drawn, so the first frames hold a thermal transient.
- **in-place** (`swap_coupling_in_context`) — a true instantaneous H_A→H_B
  switch with velocity continuity, *if* OpenMiChroM exposes the coupling as a
  swappable tabulated function.

`verify_inplace_quench` answers which you get, by swapping in a live Context and
checking whether the potential energy actually changed.

In [ ]:
# model B: a deliberately different potential (rank-1) so energy MUST change
m_B = Model.from_type_matrix(_A_FULL, _L_FULL, real_types=[0,1,2,3,4], k=1)

report = verify_inplace_quench(SEQ_HG38, m5, m_B, platform='cuda',
                               out_dir=f'{OUT}/_verify')
print(json.dumps(report, indent=2, default=str)[:2000])

In [ ]:
if report['supported']:
    print('IN-PLACE QUENCH SUPPORTED — use swap_coupling_in_context() for')
    print('velocity-continuous quenches.')
else:
    print('IN-PLACE QUENCH NOT AVAILABLE — use restart-based quench_replica().')
    print('Notes:'); [print('  -', n) for n in report['notes']]

### 5a. Restart-based quench (the guaranteed path)

Seed cell B from the **final frame of each cell-A replica** — one independent
start per replica. (Drawing many frames from a single replica gives correlated
starts and inflates apparent ensemble size without adding information.)

`phases=('production',)` — equilibration MUST NOT run, or the relaxation
transient being measured is destroyed.

In [ ]:
start = sim.load_trajectory('dyn', 0)['xyz'][-1]      # a cell-A configuration
print('start config shape', start.shape)

sim.quench_replica('quench', 0, model_B=None, initial_coords=start,
                   n_production=50_000,
                   save_spec=SaveSpec(contact_map=True, trajectory=True,
                                      velocities=True, interval=100))
# NOTE: model_B=None keeps the same potential (a control quench).
# For a real quench pass a different Model or types_table.

q = sim.load_trajectory('quench', 0)
qm = sim.load_metadata('quench', 0)
print('quench frames:', q['xyz'].shape, '| is_quench flag:', qm['is_quench'])
print('phases run:', qm['phases'])

## 6. Summary

One `Simulator`, one physics implementation, validated once:

- **statics** — `SAVE_MAPS_ONLY`, three phases → reproduces k2
- **dynamics** — `SAVE_DYNAMICS`, float32 positions + velocities, uniform interval
- **quench** — `initial_coords` + production-only, restart or in-place
- **any potential** — swap the `Model`; the simulator is untouched

Every run writes a metadata sidecar (integrator, γ, temperature, mass, force list,
versions, seed, convergence), so runs are reproducible and forces are
recomputable even when not saved.

**Next:** kernel optimization work (fit Λ for fixed C), now that the forward path
covers every experiment type.